In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
class DataCleaner:
    def __init__(self, data_path):
        # Dataset ko load karna
        self.df = pd.read_csv(data_path)
        
    def clean_data(self):
        # 1. Missing values ko handle karna
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                # Text wale columns mein sabse common value daalna
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            else:
                # Number wale columns mein average (median) daalna
                self.df[col] = self.df[col].fillna(self.df[col].median())
                
        # 2. Text categories ko numbers mein badalna (Label Encoding)
        le = LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                self.df[col] = le.fit_transform(self.df[col])
                
        return self.df

In [4]:
class AIPredictor:
    def __init__(self):
        # Hum Random Forest use kar rahe hain
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        
    def train_and_evaluate(self, df, target_column):
        # Features (X) aur Target (y) ko alag karna
        X = df.drop([target_column, 'PassengerId'], axis=1) # PassengerId prediction ke liye zaroori nahi
        y = df[target_column]
        
        # Data ko Train aur Test mein split karna
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Model Train karna
        print("Training AI Model...")
        self.model.fit(X_train, y_train)
        
        # Accuracy check karna
        predictions = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        print(f"Model Accuracy: {accuracy * 100:.2f}% 🚀")

In [5]:
# Yahan Kaggle ka wo path daal jo first cell mein dikhta hai
file_path = '/kaggle/input/competitions/spaceship-titanic/train.csv'

# 1. Cleaner object banaya aur data clean kiya
cleaner = DataCleaner(file_path)
clean_df = cleaner.clean_data()

# 2. Predictor object banaya aur model run kiya
predictor = AIPredictor()
# Spaceship titanic mein target column ka naam 'Transported' hai
predictor.train_and_evaluate(clean_df, 'Transported')

Training AI Model...
Model Accuracy: 78.84% 🚀
